# IFRS S1/S2 Requirements Extraction Notebook

This notebook builds a practical, traceable IFRS S1/S2 requirements knowledge base for ESG banking report-generation agents.

It extracts only requirements needed for these report sections:

- General Requirements
- Governance
- Strategy
- Risk Management
- Metrics and Targets

Design principle: the PDFs are the source of truth. The notebook does not invent requirements and does not use an LLM. It parses paragraph-labelled text, maps paragraphs to report sections using explicit auditable rules, adds lightweight metadata for validation and banking relevance, then exports JSON/JSONL/CSV files.

> Important: keep the original PDFs in the configured paths or update `SOURCE_PDFS` below.

## Recommended extraction approach

1. **Parse layout, not plain text only.** IFRS paragraph numbers are placed in the page margin. Plain text extraction can separate paragraph IDs from their text. The notebook uses PDF layout blocks and coordinates to attach each paragraph number to the correct paragraph body.

2. **Extract paragraph-level records first.** Each record stores `standard`, `paragraph_id`, `start_page`, `end_page`, and `paragraph_text`.

3. **Map paragraphs through explicit section rules.** The notebook uses auditable paragraph ranges rather than fuzzy semantic classification. This avoids hardcoding outputs while keeping the extraction traceable.

4. **Separate extraction from interpretation.** The source paragraph text is preserved for traceability. Derived fields such as `mandatory_status`, `evidence_tags`, and `banking_relevance` are rule-based metadata to help the ESG report-generation pipeline.

5. **Validate aggressively.** The notebook checks expected anchor paragraphs, missing text, duplicate references, unmapped target rows, invalid report sections, and banking/financed-emissions coverage.

6. **Export generation-ready files.** Outputs include JSON, JSONL, CSV, and an audit summary. The JSON is best for agents; CSV is best for manual review.

In [6]:
# ============================================================
# 0. Imports and environment check
# ============================================================
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError as exc:
    raise ImportError(
        "PyMuPDF is required. Install it with: pip install pymupdf"
    ) from exc

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)

In [7]:
# ============================================================
# 1. Configuration
# ============================================================
SOURCE_PDFS = {
    "IFRS S1": Path("gen_data/IFRS/ifrs_s1.pdf"),
    "IFRS S2": Path("gen_data/IFRS/ifrs_s2.pdf"),
}

OUTPUT_DIR = Path("gen_data/rquirements/ifrs_requirements_kb_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_REPORT_SECTIONS = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

for standard, pdf_path in SOURCE_PDFS.items():
    if not pdf_path.exists():
        raise FileNotFoundError(f"Missing source PDF for {standard}: {pdf_path}")

print("Source PDFs found:")
for standard, pdf_path in SOURCE_PDFS.items():
    print(f"- {standard}: {pdf_path} ({pdf_path.stat().st_size:,} bytes)")

Source PDFs found:
- IFRS S1: gen_data\IFRS\ifrs_s1.pdf (308,499 bytes)
- IFRS S2: gen_data\IFRS\ifrs_s2.pdf (301,537 bytes)


In [8]:
# ============================================================
# 2. Text normalization and paragraph-ID utilities
# ============================================================
PARAGRAPH_ID_RE = re.compile(r"^(?:[A-Z]\d+[A-Z]?|\d+[A-Z]?)$")
PARAGRAPH_ID_PARTS_RE = re.compile(r"^([A-Z]?)(\d+)([A-Z]?)$")

# Headings that should stop extraction after the final numbered paragraph.
STOP_HEADINGS = (
    "Approval by",
    "FOR THE ACCOMPANYING",
    "FOR THE BASIS",
)


def normalize_text(text: str) -> str:
    """Clean PDF extraction artefacts while preserving IFRS wording."""
    if text is None:
        return ""
    text = text.replace("\u00ad", "")  # soft hyphen
    text = text.replace("\ufffe", "")
    text = text.replace("\u2028", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    # Repair common line-break artefacts without changing meaning.
    replacements = {
        "sustainability- related": "sustainability-related",
        "climate- related": "climate-related",
        "greenhouse gas emissions disclosure emissions": "greenhouse gas emissions disclosure",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text


def paragraph_sort_key(paragraph_id: str) -> Tuple[str, int, int]:
    """Sort IDs such as 29, 29A, B58, B62A."""
    match = PARAGRAPH_ID_PARTS_RE.match(paragraph_id)
    if not match:
        raise ValueError(f"Invalid paragraph id: {paragraph_id}")
    prefix, number, suffix = match.groups()
    suffix_value = 0 if suffix == "" else (ord(suffix) - ord("A") + 1)
    return prefix, int(number), suffix_value


def paragraph_in_range(paragraph_id: str, start: str, end: str) -> bool:
    """Return True when paragraph_id is within an inclusive IFRS paragraph range."""
    p = paragraph_sort_key(paragraph_id)
    s = paragraph_sort_key(start)
    e = paragraph_sort_key(end)
    if not (p[0] == s[0] == e[0]):
        return False
    return s <= p <= e


def make_requirement_id(standard: str, paragraph_id: str) -> str:
    return f"{standard.replace(' ', '_')}.{paragraph_id}"

In [9]:
# ============================================================
# 3. PDF paragraph extraction using layout blocks
# ============================================================

def extract_paragraphs_from_pdf(pdf_path: Path, standard: str) -> pd.DataFrame:
    """
    Extract IFRS paragraph records from a PDF.

    Why layout blocks?
    - In IFRS PDFs, paragraph numbers often appear in a left margin block.
    - The paragraph body appears in a separate block on the same horizontal line.
    - This function detects paragraph-number blocks and collects body blocks until the next paragraph number.
    """
    doc = fitz.open(str(pdf_path))
    pages: List[List[dict]] = []
    events: List[dict] = []
    stop_events: List[dict] = []

    for page_index, page in enumerate(doc):
        page_blocks: List[dict] = []
        raw_blocks = page.get_text("dict").get("blocks", [])

        for block in raw_blocks:
            if block.get("type") != 0:
                continue

            text = " ".join(
                "".join(span.get("text", "") for span in line.get("spans", []))
                for line in block.get("lines", [])
            )
            text = normalize_text(text)
            if not text:
                continue

            x0, y0, x1, y1 = block["bbox"]
            is_header = y1 < 115
            is_footer = y0 > 705

            # The margin x-position alternates between odd/even pages.
            is_paragraph_label = (
                bool(PARAGRAPH_ID_RE.fullmatch(text))
                and 90 <= x0 <= 150
                and 115 <= y0 <= 700
            )
            is_stop_heading = (
                any(text.startswith(h) for h in STOP_HEADINGS)
                and 90 <= x0 <= 170
                and 115 <= y0 <= 700
            )

            record = {
                "standard": standard,
                "page": page_index + 1,
                "page_index": page_index,
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text,
                "is_header": is_header,
                "is_footer": is_footer,
                "is_paragraph_label": is_paragraph_label,
                "is_stop_heading": is_stop_heading,
            }
            page_blocks.append(record)

            if is_paragraph_label:
                events.append({
                    "standard": standard,
                    "paragraph_id": text,
                    "page": page_index + 1,
                    "page_index": page_index,
                    "y0": y0,
                })

            if is_stop_heading:
                stop_events.append({"page_index": page_index, "y0": y0, "text": text})

        pages.append(page_blocks)

    events = sorted(events, key=lambda e: (e["page_index"], e["y0"]))
    extracted: List[dict] = []

    for idx, event in enumerate(events):
        next_event = events[idx + 1] if idx + 1 < len(events) else None
        boundary_page_index = next_event["page_index"] if next_event else len(pages) - 1
        boundary_y = next_event["y0"] if next_event else 705

        # If an approval/basis/conclusion heading appears before the next paragraph label, stop there.
        for stop_event in stop_events:
            if (stop_event["page_index"], stop_event["y0"]) > (event["page_index"], event["y0"]):
                if (stop_event["page_index"], stop_event["y0"]) < (boundary_page_index, boundary_y):
                    boundary_page_index = stop_event["page_index"]
                    boundary_y = stop_event["y0"]
                break

        parts: List[str] = []
        pages_used = set()

        for page_index in range(event["page_index"], boundary_page_index + 1):
            start_y = event["y0"] - 4 if page_index == event["page_index"] else 115
            end_y = boundary_y - 1 if page_index == boundary_page_index else 705

            for block in sorted(pages[page_index], key=lambda b: (b["y0"], b["x0"])):
                if block["is_paragraph_label"] or block["is_header"] or block["is_footer"] or block["is_stop_heading"]:
                    continue

                # Body text is indented relative to paragraph labels. This excludes section headings.
                if block["x0"] < 145:
                    continue

                if start_y <= block["y0"] < end_y:
                    parts.append(block["text"])
                    pages_used.add(block["page"])

        paragraph_text = normalize_text(" ".join(parts))
        extracted.append({
            "standard": standard,
            "paragraph_id": event["paragraph_id"],
            "requirement_id": make_requirement_id(standard, event["paragraph_id"]),
            "start_page": min(pages_used) if pages_used else event["page"],
            "end_page": max(pages_used) if pages_used else event["page"],
            "paragraph_text": paragraph_text,
        })

    return pd.DataFrame(extracted)

In [10]:
# ============================================================
# 4. Auditable report-section mapping rules
# ============================================================
# These rules choose which paragraph ranges are relevant to the five target report sections.
# They are intentionally explicit so that a reviewer can adjust them without changing parser logic.

SECTION_RULES = [
    # IFRS S1 - main body
    {"standard": "IFRS S1", "start": "1", "end": "24", "report_section": "General Requirements", "topic": "Objective, scope, conceptual foundations, materiality, reporting entity, connected information"},
    {"standard": "IFRS S1", "start": "25", "end": "25", "report_section": "General Requirements", "topic": "Core content categories"},
    {"standard": "IFRS S1", "start": "26", "end": "27", "report_section": "Governance", "topic": "Sustainability-related governance"},
    {"standard": "IFRS S1", "start": "28", "end": "42", "report_section": "Strategy", "topic": "Sustainability-related strategy"},
    {"standard": "IFRS S1", "start": "43", "end": "44", "report_section": "Risk Management", "topic": "Sustainability-related risk management"},
    {"standard": "IFRS S1", "start": "45", "end": "53", "report_section": "Metrics and Targets", "topic": "Sustainability-related metrics and targets"},
    {"standard": "IFRS S1", "start": "54", "end": "86", "report_section": "General Requirements", "topic": "Sources of guidance, location, timing, comparative information, compliance, judgements, uncertainty, errors"},

    # IFRS S1 - selected application guidance useful for validation and generation guardrails
    {"standard": "IFRS S1", "start": "B1", "end": "B12", "report_section": "General Requirements", "topic": "Identifying sustainability-related risks and opportunities"},
    {"standard": "IFRS S1", "start": "B13", "end": "B37", "report_section": "General Requirements", "topic": "Materiality, aggregation, law/regulation, commercially sensitive information"},
    {"standard": "IFRS S1", "start": "B38", "end": "B38", "report_section": "General Requirements", "topic": "Reporting entity guidance"},
    {"standard": "IFRS S1", "start": "B39", "end": "B44", "report_section": "General Requirements", "topic": "Connected information guidance"},
    {"standard": "IFRS S1", "start": "B45", "end": "B48", "report_section": "General Requirements", "topic": "Cross-reference and interim reporting guidance"},
    {"standard": "IFRS S1", "start": "B49", "end": "B54", "report_section": "Metrics and Targets", "topic": "Comparative information for metrics"},
    {"standard": "IFRS S1", "start": "B55", "end": "B59", "report_section": "General Requirements", "topic": "Error correction guidance"},
    {"standard": "IFRS S1", "start": "C1", "end": "C3", "report_section": "General Requirements", "topic": "Other sources of guidance"},
    {"standard": "IFRS S1", "start": "D1", "end": "D33", "report_section": "General Requirements", "topic": "Qualitative characteristics of useful sustainability-related financial information"},
    {"standard": "IFRS S1", "start": "E1", "end": "E6", "report_section": "General Requirements", "topic": "Effective date and transition"},

    # IFRS S2 - main body
    {"standard": "IFRS S2", "start": "1", "end": "4", "report_section": "General Requirements", "topic": "Climate objective and scope"},
    {"standard": "IFRS S2", "start": "5", "end": "7", "report_section": "Governance", "topic": "Climate-related governance"},
    {"standard": "IFRS S2", "start": "8", "end": "23", "report_section": "Strategy", "topic": "Climate-related strategy and resilience"},
    {"standard": "IFRS S2", "start": "24", "end": "26", "report_section": "Risk Management", "topic": "Climate-related risk management"},
    {"standard": "IFRS S2", "start": "27", "end": "37", "report_section": "Metrics and Targets", "topic": "Climate-related metrics and targets"},

    # IFRS S2 - application guidance
    {"standard": "IFRS S2", "start": "B1", "end": "B18", "report_section": "Strategy", "topic": "Climate resilience and scenario analysis guidance"},
    {"standard": "IFRS S2", "start": "B19", "end": "B57", "report_section": "Metrics and Targets", "topic": "Greenhouse gas emissions and Scope 3 measurement guidance"},
    {"standard": "IFRS S2", "start": "B58", "end": "B63A", "report_section": "Metrics and Targets", "topic": "Financed emissions for financial activities"},
    {"standard": "IFRS S2", "start": "B64", "end": "B65", "report_section": "Metrics and Targets", "topic": "Cross-industry metric categories guidance"},
    {"standard": "IFRS S2", "start": "B66", "end": "B71", "report_section": "Metrics and Targets", "topic": "Climate targets and carbon credits guidance"},
    {"standard": "IFRS S2", "start": "C1", "end": "C6", "report_section": "General Requirements", "topic": "Effective date and transition"},
]


def map_to_report_section(standard: str, paragraph_id: str) -> Tuple[Optional[str], Optional[str]]:
    for rule in SECTION_RULES:
        if rule["standard"] == standard and paragraph_in_range(paragraph_id, rule["start"], rule["end"]):
            return rule["report_section"], rule["topic"]
    return None, None

In [11]:
# ============================================================
# 5. Rule-based metadata tagging
# ============================================================
MANDATORY_PATTERNS = [
    r"\bshall\b",
    r"\bis required to\b",
    r"\brequires an entity to\b",
    r"\brequired by\b",
]

RELIEF_PATTERNS = [
    r"\bneed not\b",
    r"\bpermitted to\b",
    r"\bmay\b",
    r"\brelief\b",
    r"\bexemption\b",
]

BANKING_TERMS = [
    "bank", "banking", "commercial banking", "asset management", "insurance",
    "financed emissions", "loans", "loan commitments", "undrawn loan commitments",
    "project finance", "bonds", "equity investments", "counterparties", "credit risk",
    "gross exposure", "assets under management", "aum",
]

EVIDENCE_TAG_RULES = {
    "governance_body": ["governance body", "board", "committee", "charged with governance"],
    "management_role": ["management’s role", "management-level", "management uses controls"],
    "skills_competencies": ["skills", "competencies", "capabilities"],
    "remuneration": ["remuneration"],
    "strategy": ["strategy", "decision-making", "strategic"],
    "business_model": ["business model"],
    "value_chain": ["value chain"],
    "transition_plan": ["transition plan"],
    "financial_effects": ["financial position", "financial performance", "cash flows", "financial planning"],
    "resilience": ["resilience", "scenario analysis"],
    "risk_process": ["identify", "assess", "prioritise", "monitor", "risk management"],
    "metrics": ["metric", "metrics"],
    "targets": ["target", "targets"],
    "scope_1": ["scope 1"],
    "scope_2": ["scope 2"],
    "scope_3": ["scope 3"],
    "ghg_emissions": ["greenhouse gas", "ghg", "CO2 equivalent", "emissions"],
    "financed_emissions": ["financed emissions", "category 15", "investments"],
    "commercial_banking": ["commercial banking", "gross exposure", "undrawn loan commitments"],
    "methodology": ["methodology", "method", "measurement approach", "inputs", "assumptions"],
    "data_quality": ["verified", "primary data", "secondary data", "reasonable and supportable"],
    "carbon_credits": ["carbon credits", "offset"],
    "comparatives": ["comparative", "preceding period"],
    "materiality": ["material", "materiality", "omitting", "misstating", "obscuring"],
    "compliance": ["compliance", "comply", "statement of compliance"],
}


def contains_any(text_lower: str, patterns: Sequence[str]) -> bool:
    return any(re.search(pattern, text_lower) for pattern in patterns)


def classify_mandatory_status(text: str) -> str:
    text_lower = text.lower()
    has_mandatory = contains_any(text_lower, MANDATORY_PATTERNS)
    has_relief = contains_any(text_lower, RELIEF_PATTERNS)

    if has_mandatory and has_relief:
        return "conditional_or_relief_with_mandatory_conditions"
    if has_mandatory:
        return "mandatory"
    if has_relief:
        return "optional_or_relief"
    if "objective" in text_lower:
        return "objective"
    return "context_or_guidance"


def detect_evidence_tags(text: str) -> List[str]:
    text_lower = text.lower()
    tags = []
    for tag, keywords in EVIDENCE_TAG_RULES.items():
        if any(keyword.lower() in text_lower for keyword in keywords):
            tags.append(tag)
    return sorted(set(tags))


def detect_banking_relevance(text: str) -> str:
    text_lower = text.lower()
    hits = [term for term in BANKING_TERMS if term in text_lower]
    if any(term in text_lower for term in ["commercial banking", "financed emissions", "gross exposure", "undrawn loan commitments"]):
        return "high_banking_specific"
    if hits:
        return "banking_relevant"
    return "general"


def extract_subclause_markers(text: str) -> List[str]:
    # Captures markers such as (a), (b), (i), (ii), (1), (2).
    markers = re.findall(r"(?<!\w)\(([a-z]|[ivx]+|\d+)\)", text, flags=re.IGNORECASE)
    normalized = [f"({m})" for m in markers]
    return normalized


def make_requirement_label(text: str, max_chars: int = 280) -> str:
    """Create a compact extracted label from the paragraph text, without using generative rewriting."""
    text = normalize_text(text)
    if len(text) <= max_chars:
        return text
    # Prefer first sentence/introductory clause when available.
    for separator in [":", "."]:
        cut = text.find(separator)
        if 80 <= cut <= max_chars:
            return text[: cut + 1]
    return text[:max_chars].rstrip() + "..."

In [12]:
# ============================================================
# 6. Extract all paragraphs from source PDFs
# ============================================================
all_paragraphs = []
for standard, pdf_path in SOURCE_PDFS.items():
    df = extract_paragraphs_from_pdf(pdf_path, standard)
    all_paragraphs.append(df)

paragraphs_df = pd.concat(all_paragraphs, ignore_index=True)
paragraphs_df = paragraphs_df.sort_values(
    by=["standard", "paragraph_id"],
    key=lambda col: col.map(lambda x: paragraph_sort_key(x) if PARAGRAPH_ID_RE.match(str(x)) else ("Z", 9999, 9999)) if col.name == "paragraph_id" else col,
).reset_index(drop=True)

print("Extracted paragraph records:", len(paragraphs_df))
display(paragraphs_df.groupby("standard")["paragraph_id"].count().rename("paragraph_count").reset_index())
display(paragraphs_df.head(10))

Extracted paragraph records: 308


,standard,paragraph_count
0,IFRS S1,187
1,IFRS S2,121


,standard,paragraph_id,requirement_id,start_page,end_page,paragraph_text
0,IFRS S1,1,IFRS_S1.1,7,7,The objective of IFRS S1 General Requirements for Disclosure of Sustainability- related Financial Information is to require an entity to disclose information about its sustaina...
1,IFRS S1,2,IFRS_S1.2,7,7,"Information about sustainability-related risks and opportunities is useful to primary users because an entity’s ability to generate cash flows over the short, medium and long t..."
2,IFRS S1,3,IFRS_S1.3,7,7,This Standard requires an entity to disclose information about all sustainability-related risks and opportunities that could reasonably be expected to affect the entity’s cash ...
3,IFRS S1,4,IFRS_S1.4,7,7,This Standard also prescribes how an entity prepares and reports its sustainability-related financial disclosures. It sets out general requirements for the content and presenta...
4,IFRS S1,5,IFRS_S1.5,7,7,An entity shall apply this Standard in preparing and reporting sustainability-related financial disclosures in accordance with IFRS Sustainability Disclosure Standards.
5,IFRS S1,6,IFRS_S1.6,7,7,Sustainability-related risks and opportunities that could not reasonably be expected to affect an entity’s prospects are outside the scope of this Standard.
6,IFRS S1,7,IFRS_S1.7,7,7,Other IFRS Sustainability Disclosure Standards specify information an entity is required to disclose about specific sustainability-related risks and opportunities.
7,IFRS S1,8,IFRS_S1.8,8,8,An entity may apply IFRS Sustainability Disclosure Standards irrespective of whether the entity’s related general purpose financial statements (referred to as ‘financial statem...
8,IFRS S1,9,IFRS_S1.9,8,8,"This Standard uses terminology suitable for profit-oriented entities, including public-sector business entities. If entities with not-for-profit activities in the private secto..."
9,IFRS S1,10,IFRS_S1.10,8,8,"For sustainability-related financial information to be useful, it must be relevant and faithfully represent what it purports to represent. These are fundamental qualitative cha..."


In [13]:
# ============================================================
# 7. Build the requirements knowledge base
# ============================================================
kb_rows = []

for row in paragraphs_df.to_dict(orient="records"):
    report_section, topic = map_to_report_section(row["standard"], row["paragraph_id"])
    if report_section not in TARGET_REPORT_SECTIONS:
        continue

    text = row["paragraph_text"]
    kb_rows.append({
        "requirement_id": row["requirement_id"],
        "standard": row["standard"],
        "paragraph_id": row["paragraph_id"],
        "paragraph_ref": f"{row['standard']} {row['paragraph_id']}",
        "source_pdf": str(SOURCE_PDFS[row["standard"]]),
        "source_pages": f"{row['start_page']}" if row["start_page"] == row["end_page"] else f"{row['start_page']}-{row['end_page']}",
        "report_section": report_section,
        "topic": topic,
        "mandatory_status": classify_mandatory_status(text),
        "banking_relevance": detect_banking_relevance(text),
        "evidence_tags": detect_evidence_tags(text),
        "subclause_markers": extract_subclause_markers(text),
        "requirement_label": make_requirement_label(text),
        "paragraph_text": text,
    })

requirements_df = pd.DataFrame(kb_rows)

print("Requirements KB rows:", len(requirements_df))
display(requirements_df.groupby(["standard", "report_section"]).size().rename("count").reset_index())
display(requirements_df.head(10))

Requirements KB rows: 308


,standard,report_section,count
0,IFRS S1,General Requirements,153
1,IFRS S1,Governance,2
2,IFRS S1,Metrics and Targets,15
3,IFRS S1,Risk Management,2
4,IFRS S1,Strategy,15
5,IFRS S2,General Requirements,12
6,IFRS S2,Governance,3
7,IFRS S2,Metrics and Targets,69
8,IFRS S2,Risk Management,3
9,IFRS S2,Strategy,34


,requirement_id,standard,paragraph_id,paragraph_ref,source_pdf,source_pages,report_section,topic,mandatory_status,banking_relevance,evidence_tags,subclause_markers,requirement_label,paragraph_text
0,IFRS_S1.1,IFRS S1,1,IFRS S1 1,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",objective,general,[],[],The objective of IFRS S1 General Requirements for Disclosure of Sustainability- related Financial Information is to require an entity to disclose information about its sustaina...,The objective of IFRS S1 General Requirements for Disclosure of Sustainability- related Financial Information is to require an entity to disclose information about its sustaina...
1,IFRS_S1.2,IFRS S1,2,IFRS S1 2,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",context_or_guidance,general,"[financial_effects, value_chain]",[],"Information about sustainability-related risks and opportunities is useful to primary users because an entity’s ability to generate cash flows over the short, medium and long t...","Information about sustainability-related risks and opportunities is useful to primary users because an entity’s ability to generate cash flows over the short, medium and long t..."
2,IFRS_S1.3,IFRS S1,3,IFRS S1 3,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",mandatory,general,[financial_effects],[],This Standard requires an entity to disclose information about all sustainability-related risks and opportunities that could reasonably be expected to affect the entity’s cash ...,This Standard requires an entity to disclose information about all sustainability-related risks and opportunities that could reasonably be expected to affect the entity’s cash ...
3,IFRS_S1.4,IFRS S1,4,IFRS S1 4,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",context_or_guidance,general,[],[],This Standard also prescribes how an entity prepares and reports its sustainability-related financial disclosures.,This Standard also prescribes how an entity prepares and reports its sustainability-related financial disclosures. It sets out general requirements for the content and presenta...
4,IFRS_S1.5,IFRS S1,5,IFRS S1 5,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",mandatory,general,[],[],An entity shall apply this Standard in preparing and reporting sustainability-related financial disclosures in accordance with IFRS Sustainability Disclosure Standards.,An entity shall apply this Standard in preparing and reporting sustainability-related financial disclosures in accordance with IFRS Sustainability Disclosure Standards.
5,IFRS_S1.6,IFRS S1,6,IFRS S1 6,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",context_or_guidance,general,[],[],Sustainability-related risks and opportunities that could not reasonably be expected to affect an entity’s prospects are outside the scope of this Standard.,Sustainability-related risks and opportunities that could not reasonably be expected to affect an entity’s prospects are outside the scope of this Standard.
6,IFRS_S1.7,IFRS S1,7,IFRS S1 7,gen_data\IFRS\ifrs_s1.pdf,7,General Requirements,"Objective, scope, conceptual foundations, materiality, reporting entity, connected information",mandatory,general,[],[],Other IFRS Sustainability Disclosure Standards specify information an entity is required to disclose about specific sustainability-related risks and opportunities.,Other IFRS Sustainability Disclosure Standards specify information an entity is required to disclose about specific

In [14]:
# ============================================================
# 8. Validation logic
# ============================================================
EXPECTED_ANCHORS = [
    ("IFRS S1", "1"),   # S1 objective
    ("IFRS S1", "17"),  # Materiality
    ("IFRS S1", "21"),  # Connected information
    ("IFRS S1", "27"),  # Governance requirements
    ("IFRS S1", "29"),  # Strategy categories
    ("IFRS S1", "44"),  # Risk management requirements
    ("IFRS S1", "45"),  # Metrics and targets objective
    ("IFRS S1", "50"),  # Entity-developed metrics
    ("IFRS S1", "51"),  # Targets
    ("IFRS S1", "54"),  # Sources of guidance
    ("IFRS S1", "70"),  # Comparative information
    ("IFRS S2", "1"),   # S2 objective
    ("IFRS S2", "6"),   # Climate governance
    ("IFRS S2", "10"),  # Climate risks/opportunities
    ("IFRS S2", "14"),  # Transition plan
    ("IFRS S2", "22"),  # Climate resilience
    ("IFRS S2", "25"),  # Climate risk management
    ("IFRS S2", "29"),  # Cross-industry metrics and GHG emissions
    ("IFRS S2", "29A"), # 2025 financed-emissions limitation
    ("IFRS S2", "29B"), # 2025 limitation disclosure
    ("IFRS S2", "29C"), # Category 15 / financed emissions subtotal
    ("IFRS S2", "33"),  # Climate targets
    ("IFRS S2", "36"),  # GHG emissions targets
    ("IFRS S2", "B58"), # Financed emissions context
    ("IFRS S2", "B62"), # Commercial banking financed emissions
    ("IFRS S2", "B62A"),# Commercial banking disaggregation
]


def validate_requirements_kb(requirements: pd.DataFrame, paragraphs: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add_check(name: str, passed: bool, details: str = ""):
        checks.append({"check": name, "passed": bool(passed), "details": details})

    add_check(
        "No empty paragraph text",
        requirements["paragraph_text"].fillna("").str.len().gt(0).all(),
        f"empty_count={int(requirements['paragraph_text'].fillna('').str.len().eq(0).sum())}",
    )
    add_check(
        "Unique requirement_id",
        not requirements["requirement_id"].duplicated().any(),
        f"duplicate_count={int(requirements['requirement_id'].duplicated().sum())}",
    )
    add_check(
        "Only target report sections",
        set(requirements["report_section"].dropna()).issubset(set(TARGET_REPORT_SECTIONS)),
        f"sections={sorted(requirements['report_section'].dropna().unique().tolist())}",
    )
    add_check(
        "All rows have source pages",
        requirements["source_pages"].fillna("").str.len().gt(0).all(),
        "",
    )

    available = set(zip(paragraphs["standard"], paragraphs["paragraph_id"]))
    missing_anchors = [f"{std} {pid}" for std, pid in EXPECTED_ANCHORS if (std, pid) not in available]
    add_check(
        "Expected anchor paragraphs extracted",
        len(missing_anchors) == 0,
        "missing=" + ", ".join(missing_anchors) if missing_anchors else "all expected anchors present",
    )

    kb_available = set(zip(requirements["standard"], requirements["paragraph_id"]))
    missing_kb_anchors = [f"{std} {pid}" for std, pid in EXPECTED_ANCHORS if (std, pid) not in kb_available]
    add_check(
        "Expected anchor paragraphs included in KB",
        len(missing_kb_anchors) == 0,
        "missing=" + ", ".join(missing_kb_anchors) if missing_kb_anchors else "all expected anchors included",
    )

    banking_specific = requirements[requirements["banking_relevance"].eq("high_banking_specific")]
    add_check(
        "Banking-specific financed-emissions requirements present",
        len(banking_specific) > 0,
        f"high_banking_specific_count={len(banking_specific)}",
    )

    mandatory_count = requirements[requirements["mandatory_status"].str.contains("mandatory", na=False)].shape[0]
    add_check(
        "Mandatory requirements detected",
        mandatory_count > 0,
        f"mandatory_or_conditional_count={mandatory_count}",
    )

    return pd.DataFrame(checks)

validation_df = validate_requirements_kb(requirements_df, paragraphs_df)
display(validation_df)

if not validation_df["passed"].all():
    failed = validation_df[~validation_df["passed"]]
    raise ValueError("Validation failed. Review failed checks above.")
else:
    print("All validation checks passed.")

,check,passed,details
0,No empty paragraph text,True,empty_count=0
1,Unique requirement_id,True,duplicate_count=0
2,Only target report sections,True,"sections=['General Requirements', 'Governance', 'Metrics and Targets', 'Risk Management', 'Strategy']"
3,All rows have source pages,True,
4,Expected anchor paragraphs extracted,True,all expected anchors present
5,Expected anchor paragraphs included in KB,True,all expected anchors included
6,Banking-specific financed-emissions requirements present,True,high_banking_specific_count=15
7,Mandatory requirements detected,True,mandatory_or_conditional_count=203


All validation checks passed.


In [15]:
# ============================================================
# 9. Review views for notebook users
# ============================================================
section_summary = (
    requirements_df
    .groupby(["standard", "report_section", "mandatory_status", "banking_relevance"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["standard", "report_section", "mandatory_status", "banking_relevance"])
)

display(section_summary)

print("Banking / financed-emissions view:")
banking_view = requirements_df[
    requirements_df["banking_relevance"].isin(["high_banking_specific", "banking_relevant"])
][[
    "standard", "paragraph_id", "requirement_id", "paragraph_ref", "source_pages", "report_section", "topic",
    "mandatory_status", "banking_relevance", "evidence_tags", "requirement_label"
]].sort_values(["standard", "paragraph_id"])

display(banking_view)

,standard,report_section,mandatory_status,banking_relevance,count
0,IFRS S1,General Requirements,conditional_or_relief_with_mandatory_conditions,general,14
1,IFRS S1,General Requirements,context_or_guidance,banking_relevant,1
2,IFRS S1,General Requirements,context_or_guidance,general,47
3,IFRS S1,General Requirements,mandatory,banking_relevant,1
4,IFRS S1,General Requirements,mandatory,general,70
5,IFRS S1,General Requirements,objective,general,4
6,IFRS S1,General Requirements,optional_or_relief,general,16
7,IFRS S1,Governance,mandatory,general,1
8,IFRS S1,Governance,objective,general,1
9,IFRS S1,Metrics and Targets,mandatory,general,14


Banking / financed-emissions view:


,standard,paragraph_id,requirement_id,paragraph_ref,source_pages,report_section,topic,mandatory_status,banking_relevance,evidence_tags,requirement_label
85,IFRS S1,86,IFRS_S1.86,IFRS S1 86,23-25,General Requirements,"Sources of guidance, location, timing, comparative information, compliance, judgements, uncertainty, errors",mandatory,banking_relevant,"[financial_effects, governance_body, materiality, metrics, risk_process, strategy, targets, value_chain]","If an entity identifies a material error in its prior period sustainability-related financial disclosures, it shall apply paragraphs B55–B59."
99,IFRS S1,B14,IFRS_S1.B14,IFRS S1 B14,29,General Requirements,"Materiality, aggregation, law/regulation, commercially sensitive information",context_or_guidance,banking_relevant,[],The decisions of primary users relate to providing resources to the entity and involve decisions about:
215,IFRS S2,29,IFRS_S2.29,IFRS S2 29,15-17,Metrics and Targets,Climate-related metrics and targets,mandatory,high_banking_specific,"[commercial_banking, financed_emissions, ghg_emissions, methodology, metrics, remuneration, resilience, risk_process, scope_1, scope_2, scope_3, strategy, value_chain]",An entity shall disclose information relevant to the cross-industry metric categories of:
216,IFRS S2,29A,IFRS_S2.29A,IFRS S2 29A,17,Metrics and Targets,Climate-related metrics and targets,optional_or_relief,high_banking_specific,"[commercial_banking, financed_emissions, ghg_emissions, scope_3]","In preparing disclosures to meet the requirement in paragraph 29(a)(i)(3), an entity is permitted to limit what it includes in its measure of Scope 3 Category 15 greenhouse gas..."
218,IFRS S2,29C,IFRS_S2.29C,IFRS S2 29C,17,Metrics and Targets,Climate-related metrics and targets,mandatory,high_banking_specific,"[financed_emissions, ghg_emissions, scope_3]","If an entity has included Category 15 greenhouse gas emissions in its measure of Scope 3 greenhouse gas emissions disclosed in accordance with paragraph 29(a)(i)(3), the entity..."
226,IFRS S2,37,IFRS_S2.37,IFRS S2 37,19-24,Metrics and Targets,Climate-related metrics and targets,mandatory,banking_relevant,"[financed_emissions, financial_effects, ghg_emissions, materiality, metrics, risk_process, scope_1, scope_2, scope_3, strategy, targets, value_chain]","In identifying and disclosing the metrics used to set and monitor progress towards reaching a target described in paragraphs 33–34, an entity shall refer to and consider the ap..."
263,IFRS S2,B37,IFRS_S2.B37,IFRS S2 B37,34,Metrics and Targets,Greenhouse gas emissions and Scope 3 measurement guidance,mandatory,high_banking_specific,"[commercial_banking, financed_emissions, ghg_emissions, scope_3]","An entity that participates in one or more financial activities associated with asset management, commercial banking and insurance shall disclose additional information about t..."
283,IFRS S2,B57,IFRS_S2.B57,IFRS S2 B57,38,Metrics and Targets,Greenhouse gas emissions and Scope 3 measurement guidance,mandatory,high_banking_specific,"[data_quality, financed_emissions, ghg_emissions, scope_3]",This Standard includes the presumption that Scope 3 greenhouse gas emissions can be estimated reliably using secondary data and industry averages.
284,IFRS S2,B58,IFRS_S2.B58,IFRS S2 B58,38,Metrics and Targets,Financed emissions for financial activities,context_or_guidance,high_banking_specific,"[financed_emissions, ghg_emissions, risk_process]",Entities participating in financial activities face risks and opportunities related to the greenhouse gas emissions associated with those activities.
285,IFRS S2,B59,IFRS_S2.B59,IFRS S2 B59,38-39,Metrics and Targets,Financed emissions for financial activities,mandatory,high_banking_specific,"[commercial_banking, financed_emissions, ghg_emissions, scope_3]","Paragraph 29(a)(i)(3) requires an entity to disclose its absolute gross Scope 3 greenhouse gas emissions generated during the reporting period, including upstream and downstrea..."


In [16]:
# ============================================================
# 10. Export files
# ============================================================
# JSON is recommended for generation agents.
# CSV is recommended for manual audit.
# JSONL is useful for vector stores / embedding pipelines.

json_path = OUTPUT_DIR / "ifrs_s1_s2_requirements_kb.json"
jsonl_path = OUTPUT_DIR / "ifrs_s1_s2_requirements_kb.jsonl"
csv_path = OUTPUT_DIR / "ifrs_s1_s2_requirements_kb.csv"
summary_path = OUTPUT_DIR / "ifrs_s1_s2_requirements_kb_audit_summary.md"
validation_path = OUTPUT_DIR / "ifrs_s1_s2_requirements_kb_validation.csv"

records = requirements_df.to_dict(orient="records")

with json_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

with jsonl_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

requirements_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
validation_df.to_csv(validation_path, index=False, encoding="utf-8-sig")

summary_md = []
summary_md.append("# IFRS S1/S2 Requirements KB Audit Summary\n")
summary_md.append(f"Total KB rows: {len(requirements_df)}\n")
summary_md.append("\n## Section counts\n")
summary_md.append(section_summary.to_markdown(index=False))
summary_md.append("\n\n## Validation checks\n")
summary_md.append(validation_df.to_markdown(index=False))
summary_md.append("\n\n## Banking-specific paragraphs\n")
summary_md.append(banking_view[["paragraph_ref", "source_pages", "topic", "mandatory_status", "requirement_label"]].to_markdown(index=False))

summary_path.write_text("\n".join(summary_md), encoding="utf-8")

print("Exported files:")
for path in [json_path, jsonl_path, csv_path, validation_path, summary_path]:
    print(f"- {path}")

Exported files:
- gen_data\rquirements\ifrs_requirements_kb_outputs\ifrs_s1_s2_requirements_kb.json
- gen_data\rquirements\ifrs_requirements_kb_outputs\ifrs_s1_s2_requirements_kb.jsonl
- gen_data\rquirements\ifrs_requirements_kb_outputs\ifrs_s1_s2_requirements_kb.csv
- gen_data\rquirements\ifrs_requirements_kb_outputs\ifrs_s1_s2_requirements_kb_validation.csv
- gen_data\rquirements\ifrs_requirements_kb_outputs\ifrs_s1_s2_requirements_kb_audit_summary.md


## Recommended downstream schema for generation agents

The exported JSON contains one object per paragraph-level requirement/guidance item:

```json
{
  "requirement_id": "IFRS_S2.B62",
  "standard": "IFRS S2",
  "paragraph_id": "B62",
  "paragraph_ref": "IFRS S2 B62",
  "source_pdf": "/mnt/data/ifrs_s2.pdf",
  "source_pages": "39-40",
  "report_section": "Metrics and Targets",
  "topic": "Financed emissions for financial activities",
  "mandatory_status": "mandatory",
  "banking_relevance": "high_banking_specific",
  "evidence_tags": ["commercial_banking", "financed_emissions", "ghg_emissions", "methodology", "scope_1", "scope_2", "scope_3"],
  "subclause_markers": ["(a)", "(b)", "(i)", "(ii)", "(c)", "(d)"],
  "requirement_label": "...short extracted label...",
  "paragraph_text": "...source paragraph text from PDF..."
}
```

For report generation, use `paragraph_ref`, `report_section`, `topic`, `mandatory_status`, `evidence_tags`, and `requirement_label` in prompts. Keep `paragraph_text` for traceability and validation, but avoid injecting too many full paragraphs into generation prompts.

## Manual review checklist

Before using the KB in production:

1. Review the section mapping rules in `SECTION_RULES`.
2. Confirm that your banking report template really needs the selected Appendix B/C/D/E guidance.
3. Review the `banking_view` table, especially IFRS S2 financed-emissions rows.
4. Run validation and fix any failed checks before exporting.
5. In the report-generation notebook, retrieve only the requirements for the target section being generated.
6. Use paragraph references in generated reports, but do not force paragraph-by-paragraph citations in the final report unless your business requirement asks for that.